# Modelagem e Experimentos

Este notebook define a série temporal e atributos para a etapa de modelagem e compara o resultado de predição entre os Modelos tal, tal e tal contra um baseline. Os resultados são avaliados sob as métricas precision, recall e F1. 

## Série Temporal

A série temporal construída é um dia `d` que será previsto, e as colunas são os atributos conhecidos no fim do dia `d-1`. Serão consideradas as seguintes informações:

- **Informação Observada**: para o dia `d`, usamos no máximo o que foi medido em `d-1`. Médias e somas móveis são calculadas sobre a série deslocada para que a janela nunca inclua o dia `d`.
- **Calendário**: dias da semana, mês e feriados são conhecidos com antecedência, e portanto serão utilizados para o dia `d`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit


processed_file = Path("../data/processed/ons_inmet_sul.parquet")
dataset = pd.read_parquet(processed_file).set_index("data").sort_index()

horizonte         = 1             # dias entre o último dado conhecido e o dia previsto
lags              = [1, 2, 3, 7]  # lags do histórico de interrupções, em dias
janelas_historico = [7, 30]       # médias móveis do histórico, em dias
janelas_chuva     = [3, 7]        # chuva, em dias
pct_treino        = 0.60          # fração da série utilizada para treino
pert_alta         = 0.95          # percentil dos MW diários; define uma alta interrupção

alvos = ["n_perturbacoes", "mw_interrompido"]
clima = ["chuva_media", "chuva_max", "temp_max_media", "temp_min_media", "rajada_max"]
feriados = ["feriado_nacional", "ponto_facultativo", "frac_estados_feriado"]
calendario = ["dia_semana", "mes"]
observadas = alvos + clima

## Atributos

- **Histórico das interrupções**: MW e número de perturbações em `d-1`, `d-2`, `d-3` e `d-7`; média de MW em 7 e 30 dias; e quantos dos últimos 7 e 30 dias tiveram alguma perturbação. Resumem a persistência do sistema e o nível recente de ocorrências.
- **Clima defasado**: chuva, temperatura e rajada do último dia conhecido, chuva acumulada em 3 e 7 dias e a maior rajada dos últimos 3 dias.
- **Feriados**: feriado nacional, ponto facultativo e fração dos estados em feriado em `d`.
- **Calendário**: dia da semana e mês de `d`.

In [ ]:
atributos = {}
grupos = {"historico": [], 
          "clima": [], 
          "feriados": feriados, 
          "calendario": calendario}

def adicionar_grupo(grupo, nome, serie):
    atributos[nome] = serie
    grupos[grupo].append(nome)

# dados conhecidos = dados observados após deslocamento
conhecidas = dataset[observadas].shift(horizonte)

for lag in lags:
    adicionar_grupo("historico", f"mw_lag{lag}", dataset["mw_interrompido"].shift(lag))
    adicionar_grupo("historico", f"n_lag{lag}", dataset["n_perturbacoes"].shift(lag))

for janela in janelas_historico:
    adicionar_grupo("historico", f"mw_media_{janela}d", conhecidas["mw_interrompido"].rolling(janela).mean())
    adicionar_grupo("historico", f"dias_interrupcao_{janela}d", conhecidas["n_perturbacoes"].clip(upper=1).rolling(janela).sum())

for coluna in clima:
    adicionar_grupo("clima", f"{coluna}_lag{horizonte}", dataset[coluna].shift(horizonte))

for janela in janelas_chuva:
    adicionar_grupo("clima", f"chuva_acum_{janela}d", conhecidas["chuva_media"].rolling(janela).sum())

adicionar_grupo("clima", "rajada_max_3d", conhecidas["rajada_max"].rolling(3).max())

atributos = pd.DataFrame(atributos).join(dataset[calendario + feriados])

for nome, colunas in grupos.items():
    print(f"{nome}: {len(colunas)} atributos")

## Séries de Treino e Teste

A série inicia após 30 dias, já que não há histórico completo disponível. Os primeiros 60% dos dias do período formam o dataset de treino, e os 40% dos demais dias formam o dataset de teste.

O alvo de classificação `alvo_alta` é computado a partir de um limiar de MW calculado apenas com o treino.

In [ ]:
serie = atributos.join(dataset[alvos].rename(columns={"n_perturbacoes": "alvo_n", 
                                                      "mw_interrompido": "alvo_mw"}))

# daat_corte = data na posição de 60% dos dias do período
data_corte = dataset.index[int(len(dataset) * pct_treino)]
serie = serie.dropna()
treino = serie[serie.index < data_corte]
teste = serie[serie.index >= data_corte]

limiar_alta = treino["alvo_mw"].quantile(pert_alta)

treino = treino.assign(alvo_alta=(treino["alvo_mw"] >= limiar_alta).astype(int))
teste = teste.assign(alvo_alta=(teste["alvo_mw"] >= limiar_alta).astype(int))

print(f"Limiar de alta interrupção (P{pert_alta * 100:.0f} do treino): {limiar_alta:.1f} MW")

pd.DataFrame({
    "inicio": [treino.index.min().date(), teste.index.min().date()],
    "fim": [treino.index.max().date(), teste.index.max().date()],
    "dias": [len(treino), len(teste)],
    "dias_interrupcao": [(treino["alvo_n"] > 0).sum(), (teste["alvo_n"] > 0).sum()],
    "dias_alta": [treino["alvo_alta"].sum(), teste["alvo_alta"].sum()],
}, index=["treino", "teste"])

## Modelos

### Baseline

O baseline definido é baseado no histórico. Como o horizonte é de um dia e a granularidade é diária, é comparado o dia previsto `d` com os dias anteriores:

- `d-1`
- `d-7`
- `d-1`, `d-2` ou `d-3`

O treino é dividido em 4 rodadas com janelas expansivas, cada uma validando nos 120 dias seguintes. Esse tamanho foi escolhido para que cada rodada tenha alguns dias de alta e para que a validação sempre venha depois do treino.

As métricas utilizadas para validação são precision, recall e F1.

In [ ]:
baselines = {
    "d-1": lambda df: marcar_alta(df, ["mw_lag1"]),
    "d-7": lambda df: marcar_alta(df, ["mw_lag7"]),
    "d-1, d-2 ou d-3": lambda df: marcar_alta(df, ["mw_lag1", "mw_lag2", "mw_lag3"]),
}

# retorna 1 nos dias com interrupçẽos que ultrapassaram o limiar de MW interrompidos
def marcar_alta(df, colunas):
    return (df[colunas] >= limiar_alta).any(axis=1).astype(int)

def metricas(y, previsto):
    return {"precision": precision_score(y, previsto, zero_division=0),
            "recall": recall_score(y, previsto, zero_division=0),
            "f1": f1_score(y, previsto, zero_division=0)}

cv = TimeSeriesSplit(n_splits=4, test_size=120)
linhas = []

for nome, regra in baselines.items():
    # Para cada rodada em 'n_splits'
    for rodada, (_, validacao) in enumerate(cv.split(treino), start=1):
        # Pega os 'test_size' registros na rodada atual
        registros = treino.iloc[validacao]
        linhas.append({"baseline": nome, "rodada": rodada, **metricas(registros["alvo_alta"], regra(registros))})

pd.DataFrame(linhas).groupby("baseline")[["precision", "recall", "f1"]].agg(["mean", "std"]).round(3)

Os resultados dos baselines estão na tabela acima, com a média das 4 rodadas de validação. O melhor baseline é do grupo `d-7`, com F1 de 0,120, precisão de 0,119 e recall de 0,121. A regra `d-1`, `d-2` ou `d-3` tem recall maior (0,157), mas precisão de apenas 0,058, e a regra `d-1` tem o pior desempenho (F1 de 0,036). Os dias de alta correspondem a cerca de 5% do treino (54 de 1065), e como cada rodada de validação tem poucos dias de alta, essas métricas podem variar significativamente entre rodadas.

A precisão do grupo `d-7` é mais que o dobro dessa proporção, podendo indicar um padrão semanal irregular ou ruído entre rodadas, enquanto `d-1` fica abaixo dela, ou seja, saber que houve alta no dia anterior não é suficiente para prever o dia seguinte. Portanto, o baseline escolhido de referência para os modelos é o `d-7`.